# Cancel In-Progress Fabric Jobs

Companion notebook for [`cancel_in_progress_jobs.py`](./cancel_in_progress_jobs.py).

**Inside a Fabric notebook** the script authenticates automatically via
`notebookutils.mssparkutils.credentials.getToken(...)` — no `az login` needed.

Outside Fabric (local Jupyter) the script falls back to `azure-identity`
or `az` CLI.

## 1. Load the helper

Upload `cancel_in_progress_jobs.py` to your notebook's working directory
(or `%pip install` it from a package) and import the function.

In [ ]:
from cancel_in_progress_jobs import cancel_in_progress_jobs

## 2. Dry-run preview (recommended first)

Lists in-progress jobs **without** cancelling them.

In [ ]:
summary = cancel_in_progress_jobs(
    workspace="crestshield-smartclaims-sachinsaraf",  # name OR GUID; list OK
    dry_run=True,
)
summary

## 3. Cancel everything in-progress and verify

`poll=60` waits up to 60s for each cancelled job to report a terminal status.

In [ ]:
cancel_in_progress_jobs(
    workspace="crestshield-smartclaims-sachinsaraf",
    poll=60,
)

## 4. Multi-workspace cancel

In [ ]:
cancel_in_progress_jobs(
    workspace=[
        "crestshield-smartclaims-sachinsaraf",
        "crestshield-smartclaims-secondary",
    ],
    poll=60,
    workspace_concurrency=8,
)

## 5. Continuous monitoring (mirrors the hfleitas/fabriciq pattern)

Runs for `loop_duration` minutes; re-scans every `poll_interval` seconds
and cancels any newly-started in-progress job.

In [ ]:
cancel_in_progress_jobs(
    workspace="crestshield-smartclaims-sachinsaraf",
    loop=True,
    loop_duration=30,       # minutes
    poll_interval=30,       # seconds between iterations
    sleep_interval=0.1,     # gentle pacing between API calls
    poll=45,                # per-iteration terminal-status poll
)

## 6. Targeting specific items

All filters can be combined.

In [ ]:
cancel_in_progress_jobs(
    workspace="crestshield-smartclaims-sachinsaraf",
    item=["01_Bronze*", "*Silver*"],     # name or GUID, wildcards/substring ok
    item_type=["Notebook", "DataPipeline"],
    exclude_item=["*Production*"],
    poll=60,
)

## 7. Tenant-wide fast path (Fabric Admin required)

Uses a single call to Power BI `/admin/activityevents` instead of
fanning out per workspace.

In [ ]:
cancel_in_progress_jobs(
    use_activity_events=True,
    activity_lookback=60,
    exclude_workspace=["Admin", "Fabric Admin"],
    poll=60,
)

## 8. Also stop live notebook Spark sessions

In [ ]:
cancel_in_progress_jobs(
    workspace="crestshield-smartclaims-sachinsaraf",
    stop_notebook_sessions=True,
    poll=60,
)